Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [1]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic
from tqdm import tqdm
from queue import PriorityQueue
import itertools
from itertools import product, combinations
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time
from tqdm import tqdm

In [2]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    map = rng.random(size=(size, 2))
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()

In [3]:
problem = create_problem(10, density=0.15, noise_level=10, negative_values=False)
NUM = 10
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [4]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        # path = nx.shortest_path(G, s, d, weight='weight')
        path = nx.bellman_ford_path(G, s, d, weight='weight')
        cost = nx.path_weight(G, path, weight='weight')
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    ic(s, d, path, cost)
None

ic| s: 0, d: 1, path: None, cost: inf
ic| s: 0, d: 2, path: [0, 7, 2], cost: 6118.0
ic| s: 0, d: 3, path: [0, 8, 3], cost: 12420.0
ic| s: 0, d: 4, path: [0, 4], cost: 8430.0
ic| s: 0, d: 5, path: None, cost: inf
ic| s: 0, d: 6, path: [0, 8, 6], cost: 7235.0
ic| s: 0, d: 7, path: [0, 7], cost: 831.0
ic| s: 0, d: 8, path: [0, 8], cost: 1977.0
ic| s: 0, d: 9, path: [0, 7, 2, 9], cost: 8326.0
ic| s: 1, d: 2, path: [1, 9, 7, 2], cost: 19810.0
ic| s: 1, d: 3, path: None, cost: inf
ic| s: 1, d: 4, path: None, cost: inf
ic| s: 1, d: 5, path: [1, 5], cost: 2434.0
ic| s: 1, d: 6, path: [1, 9, 6], cost: 14621.0
ic| s: 1, d: 7, path: [1, 9, 7], cost: 14523.0
ic| s: 1, d: 8, path: None, cost: inf
ic| s: 1, d: 9, path: [1, 9], cost: 6771.0
ic| s: 2, d: 3, path: None, cost: inf
ic| s: 2, d: 4, path: None, cost: inf
ic| s: 2, d: 5, path: None, cost: inf
ic| s: 2, d: 6, path: [2, 9, 6], cost: 10058.0
ic| s: 2, d: 7, path: [2, 9, 7], cost: 9960.0
ic| s: 2, d: 8, path: None, cost: inf
ic| s: 2, d: 9, pat

Bellman ford needed for Johnson

In [5]:
def bellman_ford(
    graph,
    start,
    goal,
    parent_state,
    state_cost,
):

    nodes = list(graph.nodes())
    for node in nodes:
        state_cost[node] = float("inf")
        parent_state[node] = None

    state_cost[start] = 0

    # relaxation
    for _ in range(len(nodes) - 1):
        changed = False
        for u in nodes:
            if state_cost[u] == float("inf"):
                continue
            for v in graph[u]:
                weight = graph[u][v].get("weight", 1)
                if state_cost[u] + weight < state_cost[v]:
                    state_cost[v] = state_cost[u] + weight
                    parent_state[v] = u
                    changed = True
    
        #early stopping if no changes
        if not changed:
            break
    
    # negative cylcles check
    for u in nodes:
        if state_cost[u] != float("inf"):
            for v in graph[u]:
                weight = graph[u][v].get("weight", 1)
                if state_cost[u] + weight < state_cost[v]:
                    raise ValueError("Graph contains a negative-weight cycle")
                
    # Reconstruct path
    if state_cost[goal] == float("inf"):
        return None, float("inf")
    
    path = []
    s = goal
    while s is not None:
        path.append(s)
        s = parent_state[s]
    path.reverse()
    return path, state_cost[goal]

In [6]:
def greedy_best_first_search(graph, start, goal, parent_state, state_cost, h):

    def goal_test(state):
        return state == goal

    def possible_actions(state):
        return graph[state]

    def result(state, action):
        return action

    def unit_cost(state,action):
        return graph[state][action].get("weight", 1)

    def priority_function(state):
        return h(state)

    class FrontierPQ:
        def __init__(self):
            self.pq = PriorityQueue()
            self.entry = {}

        def push(self, state, p):
            self.entry[state] = p
            self.pq.put((p, state))

        def pop(self):
            while not self.pq.empty():
                p, s = self.pq.get()
                if s in self.entry and self.entry[s] == p:
                    del self.entry[s]
                    return s
            return None

        def __contains__(self, state):
            return state in self.entry

        def __bool__(self):
            return bool(self.entry)

    frontier = FrontierPQ()
    parent_state.clear()
    state_cost.clear()

    state = start
    parent_state[state] = None
    state_cost[state] = 0
    frontier.push(state, priority_function(state))

    while state is not None and not goal_test(state):
        for a in possible_actions(state):
            new_state = result(state, a)
            cost = unit_cost(state,a)
            new_g = state_cost[state] + cost

            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                state_cost[new_state] = new_g
                frontier.push(new_state, priority_function(new_state))
            elif new_state in frontier and new_g < state_cost[new_state]:
                parent_state[new_state] = state
                state_cost[new_state] = new_g
                frontier.push(new_state, priority_function(new_state))

        if frontier:
            state = frontier.pop()
        else:
            state = None

    # reconstruct path
    if state is None:
        return None, float("inf")

    path = []
    s = state
    while s is not None:
        path.append(s)
        s = parent_state[s]

    path.reverse()
    return path, state_cost[state]



In [ ]:
def johnson_potentials(graph):

    #  potentials h(v) for each node v using Bellman-Ford from dummy source.
    # returns h as a dict[node]: potential.

    # add a dummy source `s0` that connects to every node with edge weight = 0
    dummy = "__dummy__"
    # Use a new graph structure to avoid modifying original
    import networkx as nx
    G2 = nx.DiGraph()
    for u, v, data in graph.edges(data=True):
        G2.add_edge(u, v, weight=data["weight"])
    for v in graph.nodes():
        G2.add_edge(dummy, v, weight=0.0)

    parent = {}
    cost = {}
    path, _ = bellman_ford(G2, dummy, dummy, parent, cost)
    
    if cost.get(dummy, None) == float("-inf"):
        raise ValueError("Negative cycle detected in Johnson reweighting.")
    # ends in case of negative cycle found

    # otentials h(v) = cost from dummy to v
    h = {v: cost[v] for v in graph.nodes()}
    return h


def reweight_graph(graph, h):
# use johnson potential to recompute the weights of the graph 
    import networkx as nx
    Gp = nx.DiGraph()
    for u, v, data in graph.edges(data=True):
        w = data["weight"]
        new_w = w + h[u] - h[v]
        Gp.add_edge(u, v, weight=new_w)
    return Gp

In [8]:
from queue import PriorityQueue

def astar_search(graph, start, goal, parent_state, state_cost, h):

   # priority = g(n) + h(n)

    def goal_test(state):
        return state == goal

    def possible_actions(state):
        return graph[state]

    def result(state, action):
        return action

    def unit_cost(action):
        return graph[state][action].get("weight", 1)

    def priority_function(state):
        return state_cost[state] + h(state)

    class FrontierPQ:
        def __init__(self):
            self.pq = PriorityQueue()
            self.entry = {}   # state → priority

        def push(self, state, p):
            self.entry[state] = p
            self.pq.put((p, state))

        def pop(self):
            while not self.pq.empty():
                p, s = self.pq.get()
                if s in self.entry and self.entry[s] == p:
                    del self.entry[s]
                    return s
            return None

        def __contains__(self, state):
            return state in self.entry

        def __bool__(self):
            return bool(self.entry)

    frontier = FrontierPQ()
    parent_state.clear()
    state_cost.clear()

    state = start
    parent_state[state] = None
    state_cost[state] = 0
    frontier.push(state, priority_function(state))

    while state is not None and not goal_test(state):
        for a in possible_actions(state):
            new_state = result(state, a)
            cost = graph[state][a].get("weight", 1)
            new_g = state_cost[state] + cost

            # new state discovered
            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                state_cost[new_state] = new_g
                frontier.push(new_state, priority_function(new_state))

            elif new_state in frontier and new_g < state_cost[new_state]:
                parent_state[new_state] = state
                state_cost[new_state] = new_g
                frontier.push(new_state, priority_function(new_state))

        if frontier:
            state = frontier.pop()
        else:
            state = None

    if state is None:
        return None, float("inf")

    path = []
    s = state
    while s is not None:
        path.append(s)
        s = parent_state[s]

    path.reverse()
    return path, state_cost[state]


In [10]:
import random
from itertools import product, combinations
import time
import numpy as np
import networkx as nx
from tqdm import tqdm

size = [10 , 20, 50, 100, 200]
density = [0.15, 0.5, 0.8, 1.0]
noise_level = [0.0, 0.1, 0.5, 0.8]
negative_values = [False, True]

results = []

print("Computing graphs with A* for s<500 and GBFS for s>=500")

#oter loop: graph generation
outer_iter = tqdm(list(product(size, density, noise_level, negative_values)),
                  desc="Graphs", unit="graph")

for s, d, n, negative in outer_iter:

    problem = create_problem(s, density=d, noise_level=n, negative_values=negative)
    masked = np.ma.masked_array(problem, mask=np.isinf(problem))
    G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

    try:
        if negative:
            h_original = johnson_potentials(G)

            t0 = time.perf_counter()
            G_rw = reweight_graph(G, h_original)
            reweight_time = time.perf_counter() - t0

            # new potentials of the reweighted graph
            h = johnson_potentials(G_rw)

        else:
            h = johnson_potentials(G)
            G_rw = G
            reweight_time = 0.0

    except ValueError:
        # Skip graph with negative cycle
        continue


    all_pairs = list(combinations(range(s), 2))

    if s >= 500:
        pair_count = min(s/10, len(all_pairs))   # sample
        pairs = random.sample(all_pairs, pair_count)
    else:
        pairs = all_pairs

    for source, target in pairs:

        parent_state = {}
        state_cost = {}

        heuristic = lambda node, h=h, goal=target: h[node] - h[goal]

        t_path_0 = time.perf_counter()

        try:

            if s < 200:
                algorithm = "a"

                # Networx baseline
                if negative:
                    nx_path = nx.bellman_ford_path(G, source, target, weight="weight")
                else:
                    nx_path = nx.shortest_path(G, source, target, weight="weight")
                nx_cost = nx.path_weight(G, nx_path, weight="weight")

                my_path, my_cost = astar_search(
                    G_rw, source, target, parent_state, state_cost, heuristic
                )

            else:
                algorithm = "gbfs"

                # NetworkX baseline
                if negative:
                    nx_path = nx.bellman_ford_path(G, source, target, weight="weight")
                else:
                    nx_path = nx.shortest_path(G, source, target, weight="weight")
                nx_cost = nx.path_weight(G, nx_path, weight="weight")


                my_path, my_cost = greedy_best_first_search(
                    G_rw, source, target, parent_state, state_cost, heuristic
                )

        except (nx.NetworkXNoPath, nx.NetworkXUnbounded):
            continue

        t_path_1 = time.perf_counter()

        results.append({
            "algorithm": algorithm,
            "size": s,
            "density": d,
            "noise": n,
            "negative": negative,
            "source": source,
            "target": target,
            "nx_path": nx_path,
            "nx_cost": nx_cost,
            "my_path": my_path,
            "my_cost": my_cost,
            "success": (my_path == nx_path),
            "time": round(t_path_1 - t_path_0, 4),
            "reweight_time": round(reweight_time, 4),
        })


Computing graphs with A* for s<500 and GBFS for s>=500


Graphs: 100%|██████████| 160/160 [59:34<00:00, 22.34s/graph] 


In [11]:
df = pd.DataFrame(results)
csv_path = "raw_experiment_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved raw results to: {csv_path}")

df["reweight_pct"] = df.apply(
    lambda row: (row["reweight_time"] / row["time"]) * 100 
                if row["negative"] and row["time"] > 0 else np.nan, 
    axis=1
) # check for zero division


table_time = df.groupby("algorithm").agg(
    avg_time=("time", "mean"),
    std_time=("time", "std"),
    avg_reweight_pct=("reweight_pct", "mean"),
    num_queries=("time", "count")
).reset_index()

print("\nAverage Execution Time by Algorithm")
print(table_time.to_string(index=False))


table_success = (
    df.groupby(["algorithm", "negative"])
      .agg(
          success_rate=("success", "mean"),
          total_queries=("success", "count")
      )
      .reset_index()
)

# Optional: format success_rate as percentage
table_success["success_rate"] = round(table_success["success_rate"] * 100,2)

print("\nSuccess Rate by Algorithm (GBFS split by negative edges)")
print(table_success.to_string(index=False))


Saved raw results to: raw_experiment_results.csv

Average Execution Time by Algorithm
algorithm  avg_time  std_time  avg_reweight_pct  num_queries
        a  0.001515  0.001321        148.649369       136874
     gbfs  0.008396  0.004710        148.367672       398000

Success Rate by Algorithm (GBFS split by negative edges)
algorithm  negative  success_rate  total_queries
        a     False         99.60         102492
        a      True         99.15          34382
     gbfs     False         67.37         318400
     gbfs      True         79.14          79600
